# Conjugate gradient — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred,y): return np.mean(np.asarray(pred)!=np.asarray(y))
print('setup ok')

## Minimize a quadratic using directions that do not undo each other
Conjugate gradient is steepest descent with memory for symmetric positive-definite systems. These five examples show residuals, an exact first step, conjugacy, convergence in at most n steps on a 2D quadratic, and faster progress than steepest descent on an ill-conditioned example.

In [ ]:
# 1) residual is negative gradient for Ax=b
A=np.array([[4.,1.],[1.,3.]]); b=np.array([1.,2.]); x=np.zeros(2); r=b-A@x
plt.figure(figsize=(5,3)); plt.bar(['r1','r2'],r); plt.title('residual points uphill for the linear solve')
assert np.allclose(r,[1,2])

In [ ]:
# 2) first CG step minimizes along residual direction
p=r.copy(); alpha=(r@r)/(p@A@p); x1=x+alpha*p; r1=r-alpha*(A@p)
plt.figure(figsize=(5,4)); plt.scatter([0,x1[0]],[0,x1[1]]); plt.plot([0,x1[0]],[0,x1[1]],'--'); plt.title(f'alpha={alpha:.3f}')
assert abs(alpha-0.25)<1e-12 and abs(r1@p)<1e-12

In [ ]:
# 3) next direction is A-conjugate to the first
beta=(r1@r1)/(r@r); p2=r1+beta*p
plt.figure(figsize=(5,3)); plt.bar(['p1^T A p2'],[p@A@p2]); plt.title('conjugacy is zero')
assert abs(p@A@p2)<1e-12

In [ ]:
# 4) 2D SPD system solved in two CG steps
alpha2=(r1@r1)/(p2@A@p2); x2=x1+alpha2*p2; sol=np.linalg.solve(A,b)
plt.figure(figsize=(5,4)); plt.plot([0,x1[0],x2[0]],[0,x1[1],x2[1]],'-o'); plt.scatter([sol[0]],[sol[1]],c='r'); plt.title('finite termination in n steps')
assert np.allclose(x2,sol)

In [ ]:
# 5) CG beats steepest descent on an ill-conditioned quadratic
A=np.diag([1.,50.]); b=np.array([1.,1.]); sol=np.linalg.solve(A,b)
def cg(k):
    x=np.zeros(2); r=b-A@x; p=r.copy(); err=[]
    for _ in range(k):
        a=(r@r)/(p@A@p); x=x+a*p; rn=r-a*A@p; err.append(np.linalg.norm(x-sol)); p=rn+(rn@rn)/(r@r)*p; r=rn
    return err
def sd(k):
    x=np.zeros(2); err=[]
    for _ in range(k):
        g=A@x-b; a=(g@g)/(g@A@g); x=x-a*g; err.append(np.linalg.norm(x-sol))
    return err
plt.figure(figsize=(5,3)); plt.semilogy(cg(4),'-o',label='CG'); plt.semilogy(sd(4),'-s',label='steepest'); plt.legend(); plt.title('CG finishes quickly')
assert cg(2)[-1]<1e-10 and sd(2)[-1]>1e-3